# Module 2 — Exposure Analysis

> **Governance status of this notebook**
>
> Runs on **public federal data only**, at **PUBLIC tier**, under no data-use
> agreement. That is a deliberate default, see Module 4.
>
> Federal data being public makes it legally available. It does not make an
> *analysis about a Nation* publishable by default: the aggregate product is a
> new artifact, and releasing it is a governance decision that source licensing
> does not settle. `ctx.check_publication()` blocks until sign-off is recorded.
>
> Set `NATION` and `REGION_NAME` below to retarget.

## What makes Tribal exposure analysis different

What differs is **who can act, where, and with whose money.**

Allotment under the Dawes Act (1887) and subsequent fee patenting produced a
checkerboard of tribal trust land, individually allotted trust land, and fee land
inside reservation boundaries. Response authority, funding eligibility, and
treatment permitting can change parcel to parcel. BIA fire, a Tribal fire
program, county districts, and federal agencies may all hold authority within a
single drainage.

That is an operational problem with a measurable spatial signature, and a
standard WUI analysis misses it entirely by treating the reservation as one
jurisdiction. Getting it wrong is an analytical failure, not a political
nicety, a fire crossing three ownership types may need three authorizations,
and that delay is measured in hours during initial attack.

This module also demonstrates what **cannot** be mapped here, in code.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import geopandas as gpd

from daear_toolkit import sovereignty as sv
from daear_toolkit import tribal_access as ta
from daear_toolkit import tribal_indicators as ti
from daear_toolkit import fire_access as fa

NATION = "Oglala Sioux Tribe"
REGION_NAME = "Pine Ridge"
ctx = sv.GovernanceContext(nation=NATION)

boundary = ta.get_tribal_boundary(ctx, name=REGION_NAME)
BBOX = tuple(boundary.total_bounds)
fires = gpd.clip(ta.get_fire_history(ctx, BBOX, 1984, 2024, min_acres=500), boundary)

## The land status mosaic

BIA Land Area Representations trust/allotted/fee structure.

This is the layer most often omitted from Tribal wildfire analyses and the one
that most changes the operational picture.

In [ ]:
land_status = ta.get_land_status(ctx, BBOX)
land_status = gpd.clip(land_status, boundary) if not land_status.empty else land_status
print(f"{len(land_status)} land status polygons")

status_col = next((c for c in land_status.columns
                   if any(k in c.lower() for k in ("status", "owner", "type", "class"))), None)
if status_col:
    print(f"\nStatus categories in '{status_col}':")
    print(land_status[status_col].value_counts())

fig, ax = plt.subplots(figsize=(9, 7))
if status_col:
    land_status.plot(ax=ax, column=status_col, cmap="Set3", legend=True, alpha=0.8,
                     legend_kwds={"fontsize": 8, "loc": "lower left"})
boundary.boundary.plot(ax=ax, color="black", lw=1.5)
ax.set_title(f"Land status within {REGION_NAME}")
plt.tight_layout()
plt.savefig("../outputs/02_land_status.png", dpi=150)
plt.show()

## Jurisdictional complexity

A 5 km grid, counting distinct land-status categories intersecting each cell.

High-complexity cells are where a fire is most likely to cross an authority
boundary dug initial attack, and therefore where pre-negotiated mutual-aid
agreements pay for themselves. That is a concrete, actionable output an agency
partner can use, and it comes out of the jurisdiction layer, not the fire
layer.

In [ ]:
complexity = ti.jurisdictional_complexity(land_status, status_col=status_col, grid_km=5.0)
complexity = gpd.clip(complexity, boundary)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
complexity.plot(ax=axes[0], column="n_statuses", cmap="YlOrRd", legend=True, edgecolor="white", lw=0.2)
boundary.boundary.plot(ax=axes[0], color="black", lw=1.2)
axes[0].set_title("Jurisdictional complexity\n(distinct land status categories per 5 km cell)")

complexity.plot(ax=axes[1], column="n_statuses", cmap="YlOrRd", alpha=0.55, edgecolor="none")
if not fires.empty:
    fires.plot(ax=axes[1], facecolor="none", edgecolor="black", lw=0.9)
boundary.boundary.plot(ax=axes[1], color="black", lw=1.2)
axes[1].set_title("Complexity with historical fire perimeters")
plt.tight_layout()
plt.savefig("../outputs/02_jurisdictional_complexity.png", dpi=150)
plt.show()

hi = complexity[complexity["n_statuses"] >= 3]
print(f"Cells with 3+ land status categories: {len(hi)} of {len(complexity)} ({len(hi)/max(len(complexity),1):.0%})")
print("\nThese are the cells where a single fire is most likely to cross an authority")
print("boundary during initial attack. They are the argument for pre-negotiated mutual aid.")
complexity.drop(columns="geometry").to_csv("../outputs/02_jurisdictional_complexity.csv", index=False)

## Structure exposure, broken out by land status

Structure counts alone are not actionable. "200 structures exposed" becomes
usable only when split by land status, because that determines which program can
act on which of them.

**OSM completeness caveat, and it is worse here than in the Poudre.** OSM
coverage on reservations is often substantially poorer than in adjacent
non-reservation areas (this is a mapping-effort artifact, not a settlement-pattern one).
Every count below is a lower bound, and the undercount is likely larger than
it would be off-reservation. The Tribal housing authority holds far better
records, and obtaining them is a PARTNER-tier conversation, not a download.

In [ ]:
structures = fa.get_buildings(BBOX)
structures = gpd.clip(structures, boundary) if len(structures) else structures
print(f"{len(structures)} OSM structures within the boundary (LOWER BOUND)")

exposure = ti.exposure_by_land_status(structures, land_status, status_col=status_col)
print("\nStructures by land status:")
print(exposure)

if not fires.empty:
    in_fire = gpd.sjoin(structures, fires[["geometry"]], how="inner", predicate="within")
    print(f"\nStructures within a historical fire perimeter (1984-2024): {len(in_fire)}")

exposure.to_csv("../outputs/02_structure_exposure.csv", index=False)

## What this analysis will not map

Cultural sites, sacred places, burial grounds, ceremonial locations, and plant
gathering areas are all fire-relevant they are exactly what a fire manager
would want in a values-at-risk layer. They are also the data whose disclosure
causes irreversible harm, and site location databases assembled for protective
purposes have repeatedly become targeting information for looting. That is a
documented pattern, not a hypothetical.

`CulturalResourceGuard` refuses to process datasets that appear to contain this
information. It fails closed and it scans values as well as column names.

In [ ]:
from shapely.geometry import Point

# Fabricated structural example -- these are not real locations of anything.
would_be_refused = gpd.GeoDataFrame(
    {"site_id": [1, 2, 3], "site_type": ["ceremonial", "burial", "plant gathering"]},
    geometry=[Point(-102.5, 43.2), Point(-102.6, 43.3), Point(-102.4, 43.1)],
    crs="EPSG:4326",
)

try:
    sv.CulturalResourceGuard.scan(would_be_refused)
except PermissionError as e:
    print("REFUSED:\n")
    print(e)

In [ ]:
print(sv.CulturalResourceGuard.masking_is_insufficient_because())

### The workable pattern

The Nation applies avoidance buffers on their side and shares only the resulting
**mask** a "do not treat here" polygon with no indication of why. The analysis
consumes the mask. The points never leave the Nation's control, so there is
nothing in this pipeline to leak, subpoena, or re-identify.

This is better than masking for a reason worth stating: it removes the analyst
from the decision entirely. Choosing a jitter radius is an authority-to-control
question, and an analyst picking 1 km because it seemed adequate has made a
sovereignty decision without the sovereign.

In [ ]:
# Structural illustration of consuming a mask without ever holding the points.
def apply_avoidance_mask(analysis_area_gdf, mask_gdf):
    """
    Remove Nation-designated avoidance areas from an analysis extent.

    `mask_gdf` carries geometry and nothing else, no site type, no count, no
    identifier. That absence is the design: an attribute saying WHY an area is
    masked reintroduces exactly the sensitivity the mask exists to remove.
    """
    extra = [c for c in mask_gdf.columns if c != "geometry"]
    if extra:
        raise ValueError(
            f"Avoidance mask carries attribute columns: {extra}. A mask should be geometry "
            f"only -- attributes describing why an area is masked reintroduce the sensitivity."
        )
    return gpd.overlay(analysis_area_gdf, mask_gdf, how="difference")

example_mask = gpd.GeoDataFrame(geometry=[Point(-102.5, 43.2).buffer(0.02)], crs="EPSG:4326")
analysis_extent = apply_avoidance_mask(boundary[["geometry"]], example_mask)
print(f"Analysis extent after applying an avoidance mask: "
      f"{float(analysis_extent.to_crs(epsg=5070).area.sum())/1e6:,.0f} km2 "
      f"(from {float(boundary.to_crs(epsg=5070).area.sum())/1e6:,.0f} km2)")

try:
    apply_avoidance_mask(boundary[["geometry"]], example_mask.assign(reason="ceremonial"))
except ValueError as e:
    print(f"\nMask carrying a 'reason' attribute: rejected\n  {e}")

## Summary

Land status mosaic, jurisdictional complexity on a 5 km grid, structure exposure
broken out by status, and an executable demonstration of what will not be mapped.

**The two findings a partner can act on:**

1. High-complexity cells identify where a fire is most likely to cross an
   authority boundary during initial attack a concrete case for
   pre-negotiated mutual aid.
2. Exposure split by land status shows which program can act on which structures,
   which a single reservation-wide count cannot.

**Limitations, in order of how much they matter:**

- OSM structure counts are lower bounds, and the undercount on reservations is
  likely worse than off. The Tribal housing authority holds better records; that
  is a PARTNER-tier conversation.
- BIA LAR polygons represent land status at a generalized scale and are not a
  parcel-level authority determination.
- Values at risk are incomplete by design. Cultural resources are fire-relevant
  and deliberately absent. Any "values at risk" total here undercounts what
  matters to the community, and it should say so rather than presenting itself
  as complete.